In [ ]:
%pip install -U azure-ai-ml azure-identity

In [1]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

# Connect to the Azure ML workspace
ml_client = MLClient.from_config(
    credential=DefaultAzureCredential()
)

print("Workspace name:", ml_client.workspace_name)
print("Subscription id:", ml_client.subscription_id)
print("Resource group:", ml_client.resource_group_name)

Found the config file in: /config.json


Workspace name: quick-starts-ws-303187
Subscription id: f5091c60-1c3c-430f-8d81-d802f6bf2414
Resource group: aml-quickstarts-303187


In [2]:
from azure.ai.ml.entities import AmlCompute

cpu_cluster_name = "lab-cluster-compute-notebook"

# Check whether the compute cluster already exists
try:
    compute_target = ml_client.compute.get(cpu_cluster_name)
    print(f"Found existing cluster: {cpu_cluster_name}")

except Exception:
    print("Creating a new compute cluster...")

    compute_target = AmlCompute(
        name=cpu_cluster_name,
        type="amlcompute",
        size="Standard_D2_v2",
        min_instances=0,
        max_instances=4,
        idle_time_before_scale_down=120,
    )

    compute_target = ml_client.compute.begin_create_or_update(
        compute_target
    ).result()

    print(f"Created compute cluster: {cpu_cluster_name}")

print(f"Compute target: {compute_target.name}")
print(f"VM size: {compute_target.size}")
print(f"Max nodes: {compute_target.max_instances}")

Creating a new compute cluster...
Created compute cluster: lab-cluster-compute-notebook
Compute target: lab-cluster-compute-notebook
VM size: Standard_D2_v2
Max nodes: 4


In [3]:
from azure.ai.ml.entities import Environment


conda_yaml = """
name: udacity-sklearn-env
channels:
  - conda-forge
dependencies:
  - python=3.8
  - pip
  - pip:
      - scikit-learn
      - pandas
      - numpy
      - mlflow<3
      - azureml-mlflow
"""

with open("conda.yml", "w") as f:
    f.write(conda_yaml)

print("conda.yml created.")



sklearn_env = Environment(
    name="udacity-sklearn-env",
    description="Scikit-learn environment for Udacity project",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
    conda_file="conda.yml"
)

sklearn_env = ml_client.environments.create_or_update(
    sklearn_env
)

print("Environment created:")
print("  Name:", sklearn_env.name)
print("  Version:", sklearn_env.version)

conda.yml created.
Environment created:
  Name: udacity-sklearn-env
  Version: 1


In [8]:
from azure.ai.ml import command
from azure.ai.ml.sweep import Choice, BanditPolicy

job = command(
    code="./",
    command=(
        "python train.py "
        "--C ${{inputs.C}} "
        "--max_iter ${{inputs.max_iter}}"
    ),
    environment=f"{sklearn_env.name}:{sklearn_env.version}",
    compute=cpu_cluster_name,
    inputs={
        "C": 1.0,
        "max_iter": 100
    }
)

print("Base command job created.")


sweep_job = job(
    C=Choice(values=[
        0.001,
        0.01,
        0.1,
        1,
        10,
        20,
        50,
        100,
        200,
        500,
        1000
    ]),
    max_iter=Choice(values=[
        50,
        100,
        200,
        300
    ])
).sweep(
    compute=cpu_cluster_name,
    sampling_algorithm="random",
    primary_metric="Accuracy",
    goal="maximize"
)

# Early termination
sweep_job.early_termination = BanditPolicy(
    evaluation_interval=2,
    slack_factor=0.1
)

# Limits
sweep_job.set_limits(
    max_total_trials=16,
    max_concurrent_trials=4,
    timeout=3600
)

# Naming
sweep_job.display_name = "udacity-hyperparameter-tuning"
sweep_job.experiment_name = "udacity-project"

print("Sweep job configured.")

returned_sweep_job = ml_client.jobs.create_or_update(
    sweep_job
)

print("========================================")
print("Sweep job submitted successfully!")
print("========================================")
print("Job name:", returned_sweep_job.name)
print("Studio URL:", returned_sweep_job.studio_url)

Base command job created.
Sweep job configured.
Sweep job submitted successfully!
Job name: ashy_sail_69kn9r3g69
Studio URL: https://ml.azure.com/runs/ashy_sail_69kn9r3g69?wsid=/subscriptions/f5091c60-1c3c-430f-8d81-d802f6bf2414/resourcegroups/aml-quickstarts-303187/workspaces/quick-starts-ws-303187&tid=660b3398-b80e-49d2-bc5b-ac1dc93b5254


Uploading odl_user_303187 (4.14 MBs): 100%|██████████| 4138795/4138795 [00:00<00:00, 25973813.40it/s]




In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

# Reconnect if necessary
ml_client = MLClient.from_config(
    credential=DefaultAzureCredential()
)

workspace = ml_client.workspaces.get(
    ml_client.workspace_name
)

print("Workspace:", ml_client.workspace_name)
print("MLflow tracking URI:")
print(workspace.mlflow_tracking_uri)

In [26]:
# %pip install mlflow azureml-mlflow
import mlflow

mlflow.set_tracking_uri(
    workspace.mlflow_tracking_uri
)

print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: azureml://eastus2.api.azureml.ms/mlflow/v2.0/subscriptions/f5091c60-1c3c-430f-8d81-d802f6bf2414/resourceGroups/aml-quickstarts-303187/providers/Microsoft.MachineLearningServices/workspaces/quick-starts-ws-303187


In [25]:
# ============================================================
# Find the best Accuracy run for the completed sweep
# ============================================================

import mlflow
from mlflow.tracking import MlflowClient

# ------------------------------------------------------------
# 1. Get workspace and configure MLflow
# ------------------------------------------------------------

workspace = ml_client.workspaces.get(
    ml_client.workspace_name
)

mlflow.set_tracking_uri(
    workspace.mlflow_tracking_uri
)

mlflow_client = MlflowClient()


# ------------------------------------------------------------
# 2. Get sweep
# ------------------------------------------------------------

sweep_job = ml_client.jobs.get(
    returned_sweep_job.name
)

print("Sweep:")
print("  Name:", sweep_job.name)
print("  Status:", sweep_job.status)
print("  Experiment:", sweep_job.experiment_name)


# ------------------------------------------------------------
# 3. Get completed child jobs
# ------------------------------------------------------------

children = list(
    ml_client.jobs.list(
        parent_job_name=sweep_job.name
    )
)

completed_children = [
    child
    for child in children
    if child.status == "Completed"
]

print(
    "\nCompleted Azure ML trials:",
    len(completed_children)
)


# ------------------------------------------------------------
# 4. Get MLflow experiment
# ------------------------------------------------------------

experiment = mlflow_client.get_experiment_by_name(
    sweep_job.experiment_name
)

print(
    "MLflow experiment ID:",
    experiment.experiment_id
)


# ------------------------------------------------------------
# 5. Retrieve ALL MLflow runs from this experiment
# ------------------------------------------------------------

runs = mlflow_client.search_runs(
    experiment_ids=[experiment.experiment_id],
    max_results=1000
)

print(
    "Total MLflow runs in experiment:",
    len(runs)
)


# ------------------------------------------------------------
# 6. Print all runs and their metrics
# ------------------------------------------------------------

print("\nMLflow runs:\n")

for run in runs:

    print(
        "Run ID:",
        run.info.run_id
    )

    print(
        "Run name:",
        run.info.run_name
    )

    print(
        "Accuracy:",
        run.data.metrics.get("Accuracy")
    )

    print(
        "Params:",
        run.data.params
    )

    print("--------------------------------")


# ------------------------------------------------------------
# 7. Find runs that have Accuracy
# ------------------------------------------------------------

runs_with_accuracy = [
    run
    for run in runs
    if run.data.metrics.get("Accuracy") is not None
]

print(
    "\nRuns with Accuracy:",
    len(runs_with_accuracy)
)


# ------------------------------------------------------------
# 8. Find best run
# ------------------------------------------------------------

if not runs_with_accuracy:
    raise RuntimeError(
        "MLflow contains no runs with Accuracy."
    )

best_run = max(
    runs_with_accuracy,
    key=lambda run: run.data.metrics["Accuracy"]
)


# ------------------------------------------------------------
# 9. Print best run
# ------------------------------------------------------------

print("\n==============================")
print("BEST RUN")
print("==============================")

print(
    "MLflow Run ID:",
    best_run.info.run_id
)

print(
    "Run name:",
    best_run.info.run_name
)

print(
    "Accuracy:",
    best_run.data.metrics["Accuracy"]
)

print(
    "Hyperparameters:",
    best_run.data.params
)

Sweep:
  Name: ashy_sail_69kn9r3g69
  Status: Completed
  Experiment: udacity-project

Completed Azure ML trials: 16
MLflow experiment ID: 3e259ae4-6e42-4263-90d7-e60ea38c4f31
Total MLflow runs in experiment: 17

MLflow runs:

Run ID: ashy_sail_69kn9r3g69
Run name: udacity-hyperparameter-tuning
Accuracy: None
Params: {}
--------------------------------
Run ID: ashy_sail_69kn9r3g69_1
Run name: udacity-hyperparameter-tuning_1
Accuracy: 0.9160849772382398
Params: {'C': '1.0', 'max_iter': '50'}
--------------------------------
Run ID: ashy_sail_69kn9r3g69_0
Run name: udacity-hyperparameter-tuning_0
Accuracy: None
Params: {'C': '10.0', 'max_iter': '200'}
--------------------------------
Run ID: ashy_sail_69kn9r3g69_2
Run name: udacity-hyperparameter-tuning_2
Accuracy: 0.9168437025796662
Params: {'C': '0.1', 'max_iter': '300'}
--------------------------------
Run ID: ashy_sail_69kn9r3g69_3
Run name: udacity-hyperparameter-tuning_3
Accuracy: 0.9153262518968134
Params: {'C': '0.001', 'max_iter

In [36]:
# Create TabularDataset using TabularDatasetFactory
# Data is available at: 
# "https://automlsamplenotebookdata.blob.core.windows.net/automl-sample-notebook-data/bankmarketing_train.csv"

import os

# ------------------------------------------------------------
# 1. Create an MLTable folder locally
# ------------------------------------------------------------

os.makedirs("./bankmarketing_mltable", exist_ok=True)

# Copy the CSV into the MLTable folder
import shutil

shutil.copy(
    "./bankmarketing_train.csv",
    "./bankmarketing_mltable/bankmarketing_train.csv"
)

# Create the MLTable definition
mltable_content = """
$schema: https://azuremlschemas.azureedge.net/latest/MLTable.schema.json

paths:
  - file: ./bankmarketing_train.csv

transformations:
  - read_delimited:
      delimiter: ','
      encoding: utf8
      header: all_files_same_headers
"""

with open("./bankmarketing_mltable/MLTable", "w") as f:
    f.write(mltable_content)

print("MLTable created successfully.")

MLTable created successfully.


In [38]:
from azure.ai.ml import automl, Input
from azure.ai.ml.constants import AssetTypes

# ------------------------------------------------------------
# 2. Define training data as an MLTable
# ------------------------------------------------------------

training_data = Input(
    type=AssetTypes.MLTABLE,
    path="./bankmarketing_mltable"
)

# ------------------------------------------------------------
# 3. Create AutoML classification job
# ------------------------------------------------------------

automl_job = automl.classification(
    compute=cpu_cluster_name,
    experiment_name="udacity-project-automl",
    training_data=training_data,
    target_column_name="y",
    primary_metric="accuracy",
    n_cross_validations=5
)

# ------------------------------------------------------------
# 4. Set the 30-minute limit required by Udacity
# ------------------------------------------------------------

automl_job.set_limits(
    timeout_minutes=30
)

automl_job.display_name = "udacity-automl"

# ------------------------------------------------------------
# 5. Submit
# ------------------------------------------------------------

returned_automl_job = ml_client.jobs.create_or_update(
    automl_job
)

print("========================================")
print("AutoML job submitted successfully!")
print("========================================")
print("Job name:", returned_automl_job.name)
print("Studio URL:", returned_automl_job.studio_url)

Uploading bankmarketing_mltable (4.12 MBs): 100%|██████████| 4117706/4117706 [00:00<00:00, 20326251.20it/s]




AutoML job submitted successfully!
Job name: ivory_house_93wnj8vm9z
Studio URL: https://ml.azure.com/runs/ivory_house_93wnj8vm9z?wsid=/subscriptions/f5091c60-1c3c-430f-8d81-d802f6bf2414/resourcegroups/aml-quickstarts-303187/workspaces/quick-starts-ws-303187&tid=660b3398-b80e-49d2-bc5b-ac1dc93b5254


In [45]:
# ============================================================
# Get BEST AutoML model information
# ============================================================

automl_job = ml_client.jobs.get(
    returned_automl_job.name
)

# Get all child jobs
children = list(
    ml_client.jobs.list(
        parent_job_name=automl_job.name
    )
)

# Keep completed AutoML children
completed_children = [
    child
    for child in children
    if child.status == "Completed"
]

print("AutoML status:", automl_job.status)
print("Completed trials:", len(completed_children))

# ------------------------------------------------------------
# Sort by AutoML primary metric (accuracy)
# ------------------------------------------------------------

completed_children.sort(
    key=lambda child: float(
        child.properties.get("score", 0)
    ),
    reverse=True
)

# ------------------------------------------------------------
# Best model
# ------------------------------------------------------------

best_automl_run = completed_children[0]

best_accuracy = float(
    best_automl_run.properties["score"]
)

best_algorithm = best_automl_run.properties.get(
    "run_algorithm"
)

best_preprocessor = best_automl_run.properties.get(
    "run_preprocessor"
)

# ------------------------------------------------------------
# Print all available properties of best run
# ------------------------------------------------------------

print("\n========================================")
print("BEST AUTOML RUN")
print("========================================")

print("Run ID:", best_automl_run.name)
print("Accuracy:", best_accuracy)
print("Algorithm:", best_algorithm)
print("Preprocessor:", best_preprocessor)

print("\nAll properties:")
for key, value in best_automl_run.properties.items():
    print(f"{key}: {value}")

# ------------------------------------------------------------
# Look for AUC_weighted
# ------------------------------------------------------------

auc_weighted = (
    best_automl_run.properties.get("AUC_weighted")
    or best_automl_run.properties.get("auc_weighted")
    or best_automl_run.properties.get("AUCWeighted")
)

print("\nAUC_weighted:", auc_weighted)

# ------------------------------------------------------------
# README table
# ------------------------------------------------------------

print("\n========================================")
print("README TABLE")
print("========================================")

print("| AutoML Model | |")
print("|--------------|---|")
print(f"| Run ID | `{best_automl_run.name}` |")
print(f"| Accuracy | `{best_accuracy:.6f}` |")
print(f"| AUC_weighted | `{auc_weighted}` |")
print(f"| Algorithm | `{best_algorithm}` |")

AutoML status: Completed
Completed trials: 31

BEST AUTOML RUN
Run ID: ivory_house_93wnj8vm9z_26
Accuracy: 0.9184825493171473
Algorithm: VotingEnsemble
Preprocessor: 

All properties:
runTemplate: automl_child
pipeline_id: __AutoML_Ensemble__
pipeline_spec: {"pipeline_id":"__AutoML_Ensemble__","objects":[{"module":"azureml.train.automl.ensemble","class_name":"Ensemble","spec_class":"sklearn","param_args":[],"param_kwargs":{"automl_settings":"{'task_type':'classification','primary_metric':'accuracy','verbosity':20,'ensemble_iterations':15,'ensemble_download_models_timeout_sec':300,'is_timeseries':False,'compute_target':'lab-cluster-compute-notebook','subscription_id':'f5091c60-1c3c-430f-8d81-d802f6bf2414','time_column_name':None,'grain_column_names':None}","ensemble_run_id":"ivory_house_93wnj8vm9z_26","experiment_name":"udacity-project-automl","workspace_name":"quick-starts-ws-303187","subscription_id":"f5091c60-1c3c-430f-8d81-d802f6bf2414","resource_group_name":"aml-quickstarts-303187"

In [46]:
# Delete the Azure ML compute cluster after completing the project

ml_client.compute.begin_delete(
    name=cpu_cluster_name
).result()

print(f"Compute cluster '{cpu_cluster_name}' deleted successfully.")

Compute cluster 'lab-cluster-compute-notebook' deleted successfully.
